In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
from sklearn.metrics import roc_curve
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
config = {
    'project': {
        'seed': 42,
        'num_workers': 0
    },
    'paths': {
        'data_root': r'G:\.shortcut-targets-by-id\1ua-L70AEx_VqEEKdgxjSMzr-wEX6GnxK\voice-manipulation-detection',
        'checkpoints': './models/checkpoints/transformer',
        'experiments_root': './experiments/transformer'
    },
    'data': {
        'debug_mode': False,
        'subset_fraction': 0.43,
        'max_length': 64000
    },
    'model': {
        'input_dim': 60,
        'd_model': 128,
        'nhead': 4,
        'num_layers': 3,
        'dim_feedforward': 256,
        'dropout': 0.5,
        'label_smoothing': 0.1
    },
    'training': {
        'batch_size': 32,
        'num_epochs': 15,
        'learning_rate': 0.0001,
        'weight_decay': 0.0001,
        'gradient_clip': 1.0,
        'warmup_epochs': 2,
        'optimizer': {
            'betas': [0.9, 0.98]
        },
        'scheduler': {
            'min_lr': 1e-07
        },
        'early_stopping': {
            'enabled': True,
            'patience': 5
        }
    },
    'logging': {
        'log_interval': 10
    },
    'reproducibility': {
        'deterministic': True,
        'benchmark': False
    }
}

print("Configuration: Transformer")
print(f"  d_model: {config['model']['d_model']}")
print(f"  Heads: {config['model']['nhead']}")
print(f"  Layers: {config['model']['num_layers']}")
print(f"  FFN dim: {config['model']['dim_feedforward']}")
print(f"  Dropout: {config['model']['dropout']}")
print(f"  Batch Size: {config['training']['batch_size']}")
print(f"  Learning Rate: {config['training']['learning_rate']}")

In [ ]:
class ASVspoofDataset(Dataset):
    def __init__(self, config, split='train'):
        import soundfile as sf
        import librosa
        self.sf = sf
        self.librosa = librosa
        self.config = config
        self.split = split
        self.root_dir = config['paths']['data_root']

        protocol_dir = os.path.join(
            self.root_dir, 'data', 'raw', 'ASVspoof2019', 'LA', 'LA',
            'ASVspoof2019_LA_cm_protocols'
        )

        protocol_files = {
            'train': 'ASVspoof2019.LA.cm.train.trn.txt',
            'dev': 'ASVspoof2019.LA.cm.dev.trl.txt',
            'eval': 'ASVspoof2019.LA.cm.eval.trl.txt'
        }

        protocol_path = os.path.join(protocol_dir, protocol_files[split])

        if not os.path.exists(protocol_path):
            raise FileNotFoundError(f"Protocol file not found: {protocol_path}")

        self.metadata = pd.read_csv(
            protocol_path, sep=' ', header=None,
            names=['speaker', 'filename', 'system', 'null', 'label']
        )

        if config['data'].get('subset_fraction', 1.0) < 1.0:
            fraction = config['data']['subset_fraction']
            self.metadata = self.metadata.sample(frac=fraction, random_state=42).reset_index(drop=True)
            print(f"  Using {fraction*100:.0f}% of {split} data: {len(self.metadata)} samples")

        self.audio_dir = os.path.join(
            self.root_dir, 'data', 'raw', 'ASVspoof2019', 'LA', 'LA',
            f'ASVspoof2019_LA_{split}', 'flac'
        )

        self.max_length = config['data'].get('max_length', 64000)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        audio_path = os.path.join(self.audio_dir, row['filename'] + '.flac')

        try:
            audio_data, sample_rate = self.sf.read(audio_path)
            if len(audio_data.shape) > 1:
                audio_data = audio_data.mean(axis=1)

            if sample_rate != 16000:
                audio_data = self.librosa.resample(audio_data, orig_sr=sample_rate, target_sr=16000)

            waveform = torch.from_numpy(audio_data).float().unsqueeze(0)

            if waveform.shape[1] > self.max_length:
                waveform = waveform[:, :self.max_length]
            else:
                pad = self.max_length - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, pad))
        except Exception:
            waveform = torch.zeros(1, self.max_length)

        label = 0 if row['label'] == 'bonafide' else 1

In [ ]:
class LFCCExtractor(nn.Module):
    def __init__(self, sample_rate=16000, n_lfcc=60, n_fft=512, hop_length=160):
        super().__init__()
        self.n_lfcc = n_lfcc
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.spec = T.Spectrogram(n_fft=n_fft, hop_length=hop_length, power=2.0)
        self.register_buffer('dct_mat', None)

    def _create_dct_matrix(self, n_freqs, n_lfcc):
        n = torch.arange(float(n_freqs)).unsqueeze(1)
        k = torch.arange(float(n_lfcc)).unsqueeze(0)
        dct = torch.cos(torch.pi / float(n_freqs) * (n + 0.5) * k)
        dct = dct / torch.sqrt(torch.sum(dct**2, dim=0, keepdim=True))
        return dct

    def forward(self, waveform):
        spec = self.spec(waveform).squeeze(1)
        spec = torch.log(torch.sqrt(spec) + 1e-10)
        if self.dct_mat is None:
            n_freqs = spec.shape[1]
            self.dct_mat = self._create_dct_matrix(n_freqs, self.n_lfcc).to(spec.device)
        spec = spec.transpose(1, 2)
        lfcc = torch.matmul(spec, self.dct_mat)
        return lfcc

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [ ]:
class TransformerSpoofDetector(nn.Module):
    def __init__(self, input_dim=60, d_model=128, nhead=4, num_layers=3,
                 dim_feedforward=256, dropout=0.5, num_classes=2):
        super().__init__()
        
        self.lfcc_extractor = LFCCExtractor(n_lfcc=input_dim)
        
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout)
        )
        
        self.pos_encoder = PositionalEncoding(d_model, max_len=1000, dropout=dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
            norm=nn.LayerNorm(d_model)
        )
        
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, lfcc_features):
        x = self.input_projection(lfcc_features)

        batch_size = x.size(0)
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)
        
        x = self.pos_encoder(x)
        x = self.transformer_encoder(x)
        
        cls_output = x[:, 0, :]
        output = self.classifier(cls_output)
        return output

print("Transformer model defined")

In [ ]:
class TransformerTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer, scheduler, config, device):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.config = config
        self.device = device
        
        self.best_eer = float('inf')
        self.best_accuracy = 0.0
        self.patience_counter = 0
        self.train_losses = []
        self.val_losses = []
        self.val_eers = []
        self.train_accs = []
        self.val_accs = []

    def train_epoch(self, epoch):
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        start_time = time.time()

        for batch_idx, (data, target) in enumerate(self.train_loader):
            data, target = data.to(self.device, non_blocking=True), target.to(self.device, non_blocking=True)

            self.optimizer.zero_grad()
            output = self.model(data)
            loss = self.criterion(output, target)
            loss.backward()

            if self.config['training'].get('gradient_clip'):
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config['training']['gradient_clip'])

            self.optimizer.step()

            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()

            if batch_idx % self.config['logging']['log_interval'] == 0:
                print(f"Epoch {epoch} [{batch_idx}/{len(self.train_loader)}] Loss: {total_loss/(batch_idx+1):.4f} Acc: {100*correct/total:.2f}%")

        avg_loss = total_loss / len(self.train_loader)
        final_acc = 100 * correct / total
        self.train_losses.append(avg_loss)
        self.train_accs.append(final_acc)
        print(f"Train Epoch {epoch}: Loss={avg_loss:.4f} Acc={final_acc:.2f}% Time={time.time()-start_time:.1f}s")
        return avg_loss, final_acc

    def validate(self, epoch):
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        all_scores = []
        all_labels = []

        with torch.no_grad():
            for data, target in self.val_loader:
                data, target = data.to(self.device, non_blocking=True), target.to(self.device, non_blocking=True)
                output = self.model(data)
                loss = self.criterion(output, target)

                probs = torch.softmax(output, dim=1)
                scores = probs[:, 1].cpu().numpy()

                total_loss += loss.item()
                _, predicted = torch.max(output, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

                all_scores.extend(scores)
                all_labels.extend(target.cpu().numpy())

        avg_loss = total_loss / len(self.val_loader)
        accuracy = 100 * correct / total
        eer = self.calculate_eer(np.array(all_labels), np.array(all_scores))

        self.val_losses.append(avg_loss)
        self.val_eers.append(eer)
        self.val_accs.append(accuracy)

        train_acc = self.train_accs[-1] if self.train_accs else 0
        gap = train_acc - accuracy
        status = "OVERFITTING" if gap > 10 else "OK"
        print(f"Val Epoch {epoch}: Loss={avg_loss:.4f} Acc={accuracy:.2f}% EER={eer:.4f} Gap={gap:.1f}% [{status}]")
        return avg_loss, accuracy, eer

    def calculate_eer(self, labels, scores):
        sorted_idx = np.argsort(scores)
        sorted_labels = labels[sorted_idx]
        sorted_scores = scores[sorted_idx]

        n_spoof = np.sum(labels == 1)
        n_bonafide = np.sum(labels == 0)

        min_eer = 1.0
        for threshold in sorted_scores:
            fa = np.sum((sorted_scores >= threshold) & (sorted_labels == 0)) / n_bonafide if n_bonafide > 0 else 0
            fr = np.sum((sorted_scores < threshold) & (sorted_labels == 1)) / n_spoof if n_spoof > 0 else 0
            eer = (fa + fr) / 2
            min_eer = min(min_eer, eer)

        return min_eer
        return eer

    def save_checkpoint(self, epoch, accuracy, eer):
        checkpoint_dir = Path(self.config['paths']['checkpoints'])
        checkpoint_dir.mkdir(parents=True, exist_ok=True)

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'eer': eer,
            'accuracy': accuracy,
            'config': self.config,
            'train_losses': self.train_losses,
            'val_losses': self.val_losses,
            'val_eers': self.val_eers,
        }

        epoch_path = checkpoint_dir / f"transformer_epoch_{epoch:02d}.pt"
        torch.save(checkpoint, epoch_path)
        print(f"Checkpoint saved: {epoch_path}")

        if eer < self.best_eer:
            self.best_eer = eer
            self.patience_counter = 0
            best_path = checkpoint_dir / "transformer_best_model.pt"
            torch.save(checkpoint, best_path)
            print(f"New best EER: {eer:.4f} saved to {best_path}")
        else:
            self.patience_counter += 1

        if accuracy > self.best_accuracy:
            self.best_accuracy = accuracy

    def should_early_stop(self):
        if not self.config['training']['early_stopping']['enabled']:
            return False
        return self.patience_counter >= self.config['training']['early_stopping']['patience']

    def get_current_lr(self):
        return self.optimizer.param_groups[0]['lr']

In [ ]:
print("Loading datasets...")

train_dataset = ASVspoofDataset(config, split='train')
val_dataset = ASVspoofDataset(config, split='dev')

print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")

labels = train_dataset.metadata['label'].values
train_bonafide = (labels == 'bonafide').sum()
train_spoof = (labels == 'spoof').sum()
print(f"Train distribution: Bonafide={train_bonafide} Spoof={train_spoof}")

lfcc_extractor = LFCCExtractor(n_lfcc=60).to(device)
collate_fn = LFCCCollate(lfcc_extractor, device)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=True,
    num_workers=config['project']['num_workers'],
    pin_memory=False,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=config['training']['batch_size'],
    shuffle=False,
    num_workers=config['project']['num_workers'],
    pin_memory=False,
    collate_fn=collate_fn
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

In [ ]:
torch.manual_seed(config['project']['seed'])
np.random.seed(config['project']['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config['project']['seed'])
    torch.backends.cudnn.deterministic = config['reproducibility']['deterministic']
    torch.backends.cudnn.benchmark = config['reproducibility']['benchmark']

total = train_bonafide + train_spoof
w_bonafide = total / (2.0 * train_bonafide)
w_spoof = total / (2.0 * train_spoof)
class_weights = torch.tensor([w_bonafide, w_spoof], dtype=torch.float).to(device)
print(f"Class weights: bonafide={w_bonafide:.4f} spoof={w_spoof:.4f}")

model = TransformerSpoofDetector(
    input_dim=config['model']['input_dim'],
    d_model=config['model']['d_model'],
    nhead=config['model']['nhead'],
    num_layers=config['model']['num_layers'],
    dim_feedforward=config['model']['dim_feedforward'],
    dropout=config['model']['dropout'],
    num_classes=2
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

optimizer = optim.AdamW(
    model.parameters(),
    lr=config['training']['learning_rate'],
    weight_decay=config['training']['weight_decay'],
    betas=config['training']['optimizer']['betas']
)

warmup_epochs = config['training'].get('warmup_epochs', 2)
total_epochs = config['training']['num_epochs']

def lr_lambda(epoch):
    if epoch < warmup_epochs:
        return (epoch + 1) / warmup_epochs
    else:
        progress = (epoch - warmup_epochs) / (total_epochs - warmup_epochs)
        return max(0.5 * (1 + math.cos(math.pi * progress)), config['training']['scheduler']['min_lr'] / config['training']['learning_rate'])

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=config['model']['label_smoothing']
)

trainer = TransformerTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    config=config,
    device=device
)

print("Transformer Trainer initialized")

In [ ]:
num_epochs = config['training']['num_epochs']
print(f"Starting training: {num_epochs} epochs")
print(f"Device: {device}")
print(f"Warmup: {config['training'].get('warmup_epochs', 2)} epochs")
print(f"Early stopping patience: {config['training']['early_stopping']['patience']}")
print("="*60)

for epoch in range(1, num_epochs + 1):
    print(f"\nEpoch {epoch}/{num_epochs} | LR: {trainer.get_current_lr():.6f}")
    print("-"*60)
    
    train_loss, train_acc = trainer.train_epoch(epoch)
    val_loss, val_acc, val_eer = trainer.validate(epoch)
    trainer.save_checkpoint(epoch, val_acc, val_eer)
    scheduler.step()

    if trainer.should_early_stop():
        print(f"\nEarly stopping at epoch {epoch}")
        print(f"Best EER: {trainer.best_eer:.4f}")
        break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\n" + "="*60)
print("Training complete")
print(f"Best EER: {trainer.best_eer:.4f} ({trainer.best_eer*100:.2f}%)")
print(f"Best Accuracy: {trainer.best_accuracy:.2f}%")

In [ ]:
checkpoint_dir = Path(config['paths']['checkpoints'])
if checkpoint_dir.exists():
    print("Saved checkpoints:")
    for ckpt in sorted(checkpoint_dir.glob('*.pt')):
        size = ckpt.stat().st_size / (1024**2)
        print(f"  {ckpt.name}: {size:.1f} MB")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(trainer.train_losses, label='Train', linewidth=2)
axes[0].plot(trainer.val_losses, label='Val', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(trainer.train_accs, label='Train', linewidth=2)
axes[1].plot(trainer.val_accs, label='Val', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy %')
axes[1].set_title('Accuracy Curves')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(trainer.val_eers, color='red', linewidth=2, label='Val EER')
axes[2].axhline(y=trainer.best_eer, color='green', linestyle='--', label=f'Best: {trainer.best_eer:.4f}')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('EER')
axes[2].set_title('Equal Error Rate')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()

fig_dir = Path(config['paths']['experiments_root']) / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / 'transformer_training_curves.png', dpi=150)
print(f"Saved: {fig_dir / 'transformer_training_curves.png'}")
plt.show()

In [ ]:
print("Transformer Training Summary")
print("="*60)
print(f"Architecture: Transformer Encoder with CLS token")
print(f"d_model: {config['model']['d_model']}")
print(f"Heads: {config['model']['nhead']}")
print(f"Layers: {config['model']['num_layers']}")
print(f"FFN dim: {config['model']['dim_feedforward']}")
print(f"Parameters: {total_params:,}")
print(f"Best EER: {trainer.best_eer:.4f}")
print(f"Best Accuracy: {trainer.best_accuracy:.2f}%")
print(f"Model saved: {config['paths']['checkpoints']}/transformer_best_model.pt")